# MedSAM2 — DCA1 Evaluation: Zero-Shot vs Fine-Tuned

**Experiment:** Proper 80/20 train/test split on DCA1.  
Compare MedSAM2 zero-shot (no training) vs fine-tuned (trained on train split).  
Metric: **Dice + IoU on 27 held-out test images** — real numbers, not vibes.

**Bucket:** `gs://coronary-angio-v2`

## Cell 1 — Environment Setup

Installs PyTorch pinned to CUDA 12.1 (`cu121`). This is **required** for the Tesla P4 GPU on this instance.  
The P4 is Pascal architecture (compute capability sm\_61). CUDA 13.0 dropped sm\_61 support, so cu121 is the last compatible build.

In [ ]:
import os, subprocess, sys

subprocess.run(['sudo','apt-get','update','-q'], check=True)
subprocess.run(['sudo','apt-get','install','-y','ffmpeg','p7zip-full'], check=True)

subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.5.1','torchvision==0.20.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'numpy>=2.0.1','opencv-python','pillow','matplotlib','tqdm','scipy','-q'], check=True)

if not os.path.exists('/home/jupyter/MedSAM2'):
    subprocess.run(['git','clone','https://github.com/bowang-lab/MedSAM2.git',
                    '/home/jupyter/MedSAM2'], check=True)

try:
    import sam2
    print('sam2 already importable')
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-e','.', '-q'],
                   cwd='/home/jupyter/MedSAM2', check=True)

sys.path.insert(0, '/home/jupyter/MedSAM2')
print('Environment ready')

## Cell 2 — Download MedSAM2 Checkpoint

**What is a checkpoint?**  
A `.pt` file containing the model's learned weights — ~150M numbers that encode everything MedSAM2 learned from millions of CT/MRI/ultrasound images. We start from this and fine-tune on DCA1.

**Config:** `sam2.1_hiera_t512.yaml` — the "tiny" variant with 512×512 input resolution.

In [ ]:
import os

BUCKET   = 'gs://coronary-angio-v2'
CKPT_DIR = '/home/jupyter/MedSAM2/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

CHECKPOINT = os.path.join(CKPT_DIR, 'MedSAM2_latest.pt')
CONFIG     = 'configs/sam2.1_hiera_t512.yaml'

if not os.path.exists(CHECKPOINT):
    subprocess.run(['bash','download.sh'], cwd='/home/jupyter/MedSAM2')
else:
    print('Checkpoint already present')

print(f'Checkpoint: {CHECKPOINT}')
print(f'Config:     {CONFIG}')

## Cell 3 — Download DCA1

DCA1: 134 static X-ray coronary angiogram images (300×300px grayscale PGM).  
Expert segmentation mask for every image (`83.pgm` → `83_gt.pgm`).  
Source: Cervantes-Sanchez et al., *Applied Sciences* 2019.

In [ ]:
import os, glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

DCA1_RAW = '/tmp/dca1/Database_134_Angiograms'
os.makedirs('/tmp/dca1', exist_ok=True)

if not os.path.exists(DCA1_RAW):
    os.system(f'gsutil -m cp -r gs://coronary-angio-v2/Database_134_Angiograms /tmp/dca1/')
else:
    print('DCA1 already on disk')

all_pgm = glob.glob(f'{DCA1_RAW}/*.pgm')
print(f'Total PGM files: {len(all_pgm)} (should be 268 = 134 images + 134 masks)')

## Cell 4 — Correct Pairing: Fix the Sort Bug

**The bug (critical):**  
`sorted(['1.pgm','10.pgm','2.pgm'])` → `['1.pgm', '10.pgm', '2.pgm']` (lexicographic order).  
`sorted(['1_gt.pgm','10_gt.pgm','2_gt.pgm'])` → `['10_gt.pgm', '1_gt.pgm', '2_gt.pgm']` (different! `_` > `.` in ASCII).

So naively zipping sorted images with sorted masks pairs **image 1 with mask 10**.

**Fix:** Sort both lists by their integer stem — `int('83')`, `int('83_gt'.replace('_gt',''))` — so the sort key is the same number for both.

In [ ]:
def img_stem(path):
    return int(os.path.basename(path).replace('.pgm',''))

def mask_stem(path):
    return int(os.path.basename(path).replace('_gt.pgm',''))

raw_images = [f for f in all_pgm if '_gt' not in f]
raw_masks  = [f for f in all_pgm if '_gt'     in f]

images_sorted = sorted(raw_images, key=img_stem)
masks_sorted  = sorted(raw_masks,  key=mask_stem)

assert len(images_sorted) == len(masks_sorted) == 134

# Verify every pair matches
mismatches = [(i, m) for i, m in zip(images_sorted, masks_sorted)
              if img_stem(i) != mask_stem(m)]
if mismatches:
    print(f'MISMATCH: {mismatches[:5]}')
else:
    print(f'All 134 pairs verified ✓')
    print(f'Sample: {os.path.basename(images_sorted[0])} ↔ {os.path.basename(masks_sorted[0])}')
    print(f'Sample: {os.path.basename(images_sorted[9])} ↔ {os.path.basename(masks_sorted[9])}')

## Cell 5 — Process DCA1: PGM → PNG

Convert PGM → RGB PNG (MedSAM2's image encoder expects 3 channels).  
Binarize masks: vessel pixels → 255, background → 0.

**One resize only:** Images saved at original 300×300 for now. We resize inside the Dataset at training time, directly to 512×512 — no double-resize.

In [ ]:
from tqdm import tqdm

IMG_PROC  = '/tmp/dca1_proc/images'
MASK_PROC = '/tmp/dca1_proc/masks'
os.makedirs(IMG_PROC,  exist_ok=True)
os.makedirs(MASK_PROC, exist_ok=True)

for img_path, msk_path in tqdm(zip(images_sorted, masks_sorted), total=134):
    stem = str(img_stem(img_path))

    img  = Image.open(img_path).convert('L')
    img_rgb = Image.fromarray(np.stack([np.array(img)]*3, axis=-1))
    img_rgb.save(os.path.join(IMG_PROC, f'{stem}.png'))

    msk = np.array(Image.open(msk_path).convert('L'))
    msk_bin = (msk > 0).astype(np.uint8) * 255
    Image.fromarray(msk_bin).save(os.path.join(MASK_PROC, f'{stem}.png'))

print(f'Processed {len(images_sorted)} pairs')

# Show foreground pixel fraction (important for loss weighting later)
fg_fracs = []
for mp in glob.glob(f'{MASK_PROC}/*.png'):
    m = np.array(Image.open(mp).convert('L').resize((256,256), Image.NEAREST))
    fg_fracs.append((m > 0).mean())
print(f'Mean foreground fraction at 256×256: {np.mean(fg_fracs):.3f}  ({np.mean(fg_fracs)*100:.1f}%)')
print(f'  → This sets pos_weight in BCE loss (background pixels ÷ foreground pixels)')

In [ ]:
# Visually verify 3 pairs — image should match its mask
proc_imgs  = sorted(glob.glob(f'{IMG_PROC}/*.png'),  key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
proc_masks = sorted(glob.glob(f'{MASK_PROC}/*.png'), key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))

fig, axes = plt.subplots(3, 2, figsize=(8, 11))
for row, idx in enumerate([0, 10, 50]):
    img = np.array(Image.open(proc_imgs[idx]))
    msk = np.array(Image.open(proc_masks[idx]))
    stem = os.path.splitext(os.path.basename(proc_imgs[idx]))[0]
    axes[row,0].imshow(img[:,:,0], cmap='gray'); axes[row,0].set_title(f'Image {stem}'); axes[row,0].axis('off')
    axes[row,1].imshow(msk, cmap='gray');        axes[row,1].set_title(f'Mask {stem}');  axes[row,1].axis('off')
plt.suptitle('Pair verification — vessels in image should match white regions in mask', fontsize=11)
plt.tight_layout()
plt.show()

## Cell 6 — Train / Test Split (80 / 20)

**Why we need this:**  
The previous experiment used all 134 images for training with no held-out test set.  
This means we had no way to measure whether the model was learning anything or just overfitting.

**80/20 split:**  
- Train: 107 images — used for fine-tuning  
- Test: 27 images — **never seen during training**, used only for final evaluation

We shuffle with a fixed random seed (42) so the split is reproducible.

In [ ]:
import random

all_stems = [int(os.path.splitext(os.path.basename(p))[0]) for p in proc_imgs]
indices   = list(range(len(all_stems)))

random.seed(42)
random.shuffle(indices)

n_train   = int(0.8 * len(indices))
train_idx = sorted(indices[:n_train])
test_idx  = sorted(indices[n_train:])

train_img_paths  = [proc_imgs[i]  for i in train_idx]
train_mask_paths = [proc_masks[i] for i in train_idx]
test_img_paths   = [proc_imgs[i]  for i in test_idx]
test_mask_paths  = [proc_masks[i] for i in test_idx]

print(f'Train: {len(train_img_paths)} images')
print(f'Test:  {len(test_img_paths)} images')
print(f'Test stems: {[int(os.path.splitext(os.path.basename(p))[0]) for p in test_img_paths]}')

## Cell 7 — Zero-Shot Evaluation on Test Split

**What "zero-shot" means here:**  
We use MedSAM2 exactly as released — no fine-tuning, no DCA1 training at all.  
We give it one click on each test image (at the centroid of the GT mask) and ask it to segment.

**Why the centroid?**  
This simulates a user clicking somewhere in the middle of the vessel.  
Both zero-shot and fine-tuned get the exact same click, so the comparison is fair.

**How SAM2ImagePredictor works:**  
1. `predictor.set_image(img_rgb)` — runs the image encoder once, caches features  
2. `predictor.predict(point_coords, point_labels)` — runs prompt encoder + mask decoder  
3. Returns a binary mask at the original image resolution

The predictor handles its own normalization internally — you pass a raw uint8 RGB array.

In [ ]:
import torch
import numpy as np
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "none"}')

def centroid_click(mask_np):
    """Return (x, y) click at the centroid of the vessel mask."""
    ys, xs = np.where(mask_np > 0)
    if len(ys) == 0:
        return (mask_np.shape[1]//2, mask_np.shape[0]//2)
    return (int(xs.mean()), int(ys.mean()))

def dice_score(pred, gt, smooth=1e-5):
    pred, gt = pred.astype(float), gt.astype(float)
    return (2*(pred*gt).sum() + smooth) / (pred.sum() + gt.sum() + smooth)

def iou_score(pred, gt, smooth=1e-5):
    i = (pred.astype(bool) & gt.astype(bool)).sum()
    u = (pred.astype(bool) | gt.astype(bool)).sum()
    return (i + smooth) / (u + smooth)

# Build zero-shot predictor
zs_model     = build_sam2(CONFIG, CHECKPOINT, device=device)
zs_predictor = SAM2ImagePredictor(zs_model)

zs_results = []  # list of {stem, dice, iou, pred_mask}
print('Running zero-shot on 27 test images...')

for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0

    cx, cy = centroid_click(gt_mask)

    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        zs_predictor.set_image(img_rgb)
        preds, scores, _ = zs_predictor.predict(
            point_coords=np.array([[cx, cy]]),
            point_labels=np.array([1]),
            multimask_output=False
        )

    pred_mask = preds[0]  # (H, W) bool
    stem = int(os.path.splitext(os.path.basename(img_path))[0])
    zs_results.append({
        'stem': stem,
        'dice': dice_score(pred_mask, gt_mask),
        'iou':  iou_score(pred_mask, gt_mask),
        'pred': pred_mask,
        'gt':   gt_mask,
        'img':  img_rgb,
    })

zs_dice = [r['dice'] for r in zs_results]
zs_iou  = [r['iou']  for r in zs_results]
print(f'\nZero-Shot Results on {len(zs_results)} test images:')
print(f'  Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f}')
print(f'  IoU:  {np.mean(zs_iou):.3f} ± {np.std(zs_iou):.3f}')

In [ ]:
# Visualize zero-shot on 6 test images (worst, median, best by Dice)
sorted_by_dice = sorted(zs_results, key=lambda r: r['dice'])
show = [sorted_by_dice[0], sorted_by_dice[len(sorted_by_dice)//4],
        sorted_by_dice[len(sorted_by_dice)//2],
        sorted_by_dice[3*len(sorted_by_dice)//4], sorted_by_dice[-2], sorted_by_dice[-1]]

fig, axes = plt.subplots(3, 6, figsize=(22, 10))
for col, r in enumerate(show):
    axes[0,col].imshow(r['img'][:,:,0], cmap='gray'); axes[0,col].set_title(f"#{r['stem']}"); axes[0,col].axis('off')
    axes[1,col].imshow(r['img'][:,:,0], cmap='gray')
    axes[1,col].imshow(r['pred'], alpha=0.45, cmap='Reds')
    axes[1,col].set_title(f"Pred  Dice={r['dice']:.2f}"); axes[1,col].axis('off')
    axes[2,col].imshow(r['gt'],  cmap='gray');  axes[2,col].set_title('GT'); axes[2,col].axis('off')

axes[0,0].set_ylabel('Original', fontsize=10)
axes[1,0].set_ylabel('ZeroShot (Red)', fontsize=10)
axes[2,0].set_ylabel('Ground Truth', fontsize=10)
plt.suptitle('Zero-Shot MedSAM2 on DCA1 Test Split (sorted worst→best Dice)', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/zeroshot_dca1.png', dpi=120)
plt.show()
!gsutil cp /tmp/zeroshot_dca1.png {BUCKET}/results/zeroshot_dca1.png
print('Saved')

## Cell 8 — Fine-Tuning Setup

**What we're doing:**  
Adjusting MedSAM2's mask decoder + prompt encoder weights on the 107 DCA1 training images,  
so it becomes better at recognising coronary vessel shapes specifically.

**What stays frozen — and why:**  
- **Image encoder (Hiera backbone, 70% of params):** Frozen. It converts pixels → features.  
  We freeze it because (a) 107 images is too few to adapt 27M params and (b) we'd need to store the gradient for every intermediate layer, which would OOM the P4's 8GB VRAM.  
- **Mask decoder + prompt encoder (30% = 11.7M params):** Trainable.  
  These map features + prompt → binary mask. This is where coronary-specific shape knowledge goes.

**Prompt type: point (centroid of GT vessel mask)**  
We use the same click strategy as zero-shot. Train and test both use centroid clicks — no mismatch.

**Loss:**  
- **Dice loss**: good for class-imbalanced segmentation (vessels are ~5% of image area).  
- **Weighted BCE** with `pos_weight = background_pixels / foreground_pixels`:  
  tells the BCE term "false negatives on vessels matter ~20× more than false positives".  
- Both losses get sigmoid applied to logits *inside* the loss function.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler

MODEL_SIZE    = 512
HIRES         = MODEL_SIZE // 4   # 128
BB_FEAT_SIZES = [[HIRES//(2**k)]*2 for k in range(3)]  # [[128,128],[64,64],[32,32]]

class DCA1Dataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        # Match SAM2's internal preprocessing (ImageNet stats, same as pretrained encoder)
        self.transform = transforms.Compose([
            transforms.Resize((MODEL_SIZE, MODEL_SIZE)),  # 300→512, single pass
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img      = Image.open(self.img_paths[idx]).convert('RGB')
        mask_pil = Image.open(self.mask_paths[idx]).convert('L')
        mask_np  = np.array(mask_pil)   # 300×300 original

        img_t = self.transform(img)     # (3, 512, 512) normalized

        # Target mask at 256×256 — SAM2 decoder outputs at this resolution
        mask_256 = np.array(mask_pil.resize((256,256), Image.NEAREST))
        mask_t   = torch.from_numpy((mask_256 > 0).astype(np.float32)).unsqueeze(0)

        # Click: random foreground pixel (simulates user clicking on vessel)
        ys, xs = np.where(mask_np > 0)
        if len(ys) > 0:
            pick = np.random.randint(len(ys))
            py, px = ys[pick], xs[pick]
        else:
            py, px = mask_np.shape[0]//2, mask_np.shape[1]//2

        h, w = mask_np.shape
        # Scale to MODEL_SIZE (512) directly — single scale factor
        point = torch.tensor([px/w*MODEL_SIZE, py/h*MODEL_SIZE], dtype=torch.float32)

        return img_t, mask_t, point

# Compute pos_weight from training masks
fg_list = []
for mp in train_mask_paths:
    m = np.array(Image.open(mp).convert('L').resize((256,256), Image.NEAREST))
    fg_list.append((m > 0).mean())
fg_mean   = np.mean(fg_list)
pos_weight = torch.tensor([(1-fg_mean)/fg_mean], device=device)
print(f'Foreground fraction: {fg_mean:.3f} ({fg_mean*100:.1f}%)')
print(f'pos_weight: {pos_weight.item():.1f}x  (background penalised {pos_weight.item():.0f}× less than foreground)')

train_ds     = DCA1Dataset(train_img_paths, train_mask_paths)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} images | {len(train_loader)} batches/epoch')

In [ ]:
from sam2.build_sam import build_sam2

model = build_sam2(CONFIG, CHECKPOINT, device=device)

# Freeze image encoder only
for name, param in model.named_parameters():
    param.requires_grad = ('image_encoder' not in name)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
print('Frozen: image encoder  |  Trainable: mask decoder + prompt encoder')

## Cell 9 — Training Loop

**How the forward pass works (bypassing the predictor):**  
`SAM2ImagePredictor.set_image()` and `.predict()` are decorated with `@torch.no_grad()`,  
which blocks gradient flow — you cannot use them for training.  
Instead we call the model's sub-components directly:  
1. `model.forward_image(img)` → backbone features  
2. `model._prepare_backbone_features(...)` → multi-scale feature maps  
3. `model.sam_prompt_encoder(points=...)` → sparse + dense prompt embeddings  
4. `model.sam_mask_decoder(...)` → low-res mask logits (256×256)

Steps 1-2 run under `torch.no_grad()` (encoder is frozen, no grad needed).  
Steps 3-4 run with grad enabled (these are the trainable parts).

**Gradient clipping:** `max_norm=1.0` — prevents exploding gradients common with cosine LR on small datasets.

In [ ]:
CKPT_PATH = '/tmp/medsam2_dca1_ft.pt'
EPOCHS    = 20

def dice_loss(logits, target, smooth=1e-5):
    pred  = torch.sigmoid(logits)   # sigmoid INSIDE the loss
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return (1 - (2*inter + smooth)/(union + smooth)).mean()

def combined_loss(logits, target, pw):
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)(logits, target)
    return dice_loss(logits, target) + bce

optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()

best_loss    = float('inf')
train_losses = []

print(f'Fine-tuning {EPOCHS} epochs on {len(train_ds)} images...')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for imgs, masks, points in tqdm(train_loader, desc=f'Ep {epoch+1}/{EPOCHS}', leave=False):
        optimizer.zero_grad()
        B = imgs.shape[0]

        imgs_dev   = imgs.to(device)
        points_dev = points.to(device)   # already in 512-space from Dataset

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            # Frozen encoder — no grad needed here
            with torch.no_grad():
                backbone_out = model.forward_image(imgs_dev)
                _, vision_feats, _, _ = model._prepare_backbone_features(backbone_out)
                if model.directly_add_no_mem_embed:
                    vision_feats[-1] = vision_feats[-1] + model.no_mem_embed

            feats = [
                feat.permute(1,2,0).view(B,-1,*fs)
                for feat, fs in zip(vision_feats[::-1], BB_FEAT_SIZES[::-1])
            ][::-1]
            image_embed    = feats[-1]
            high_res_feats = feats[:-1]

            all_logits = []
            for i in range(B):
                pt       = points_dev[i].reshape(1,1,2)
                pt_label = torch.ones(1,1, dtype=torch.int, device=device)
                sparse_emb, dense_emb = model.sam_prompt_encoder(
                    points=(pt, pt_label), boxes=None, masks=None)
                low_res_masks, _, _, _ = model.sam_mask_decoder(
                    image_embeddings=image_embed[i].unsqueeze(0),
                    image_pe=model.sam_prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_emb,
                    dense_prompt_embeddings=dense_emb,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=[f[i].unsqueeze(0) for f in high_res_feats],
                )
                all_logits.append(low_res_masks)

            logits  = torch.cat(all_logits, dim=0)
            targets = masks.to(device)
            if targets.shape[-2:] != logits.shape[-2:]:
                targets = F.interpolate(targets.float(), size=logits.shape[-2:], mode='nearest')

            loss = combined_loss(logits, targets, pos_weight)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)
    scheduler.step()
    print(f'Epoch {epoch+1:2d} | Loss: {epoch_loss:.4f}')

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'           ★ checkpoint saved (loss {best_loss:.4f})')

print(f'\nTraining complete. Best loss: {best_loss:.4f}')
!gsutil cp {CKPT_PATH} {BUCKET}/checkpoints/medsam2_dca1_ft_pointprompt.pt
print('Checkpoint uploaded to GCS')

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(1,EPOCHS+1), train_losses, 'b-o', markersize=4, label='Train Loss (Dice+wBCE)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('DCA1 Fine-Tuning Loss Curve')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/tmp/ft_loss_curve.png', dpi=120)
plt.show()
!gsutil cp /tmp/ft_loss_curve.png {BUCKET}/results/ft_loss_curve.png

## Cell 10 — Fine-Tuned Inference on Test Split

We load the fine-tuned checkpoint and wrap it in `SAM2ImagePredictor` for clean single-image inference.

**Key diagnostic: verify the checkpoint actually loaded.**  
`strict=False` silently swallows mismatched keys. We print the result and assert nothing is missing — otherwise "fine-tuned" inference is actually zero-shot with different random state.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

ft_model = build_sam2(CONFIG, CHECKPOINT, device=device)

# Load fine-tuned weights and verify
state_dict  = torch.load(CKPT_PATH, map_location=device)
incompatible = ft_model.load_state_dict(state_dict, strict=False)

print('=== Checkpoint load verification ===')
print(f'Missing keys  ({len(incompatible.missing_keys)}):   {incompatible.missing_keys[:5]}')
print(f'Unexpected keys ({len(incompatible.unexpected_keys)}): {incompatible.unexpected_keys[:5]}')
if len(incompatible.missing_keys) == 0 and len(incompatible.unexpected_keys) == 0:
    print('✓ Clean load — all fine-tuned weights applied correctly')
else:
    print('⚠ Some keys did not match — review above before trusting results')

ft_model.eval()
ft_predictor = SAM2ImagePredictor(ft_model)

ft_results = []
print('\nRunning fine-tuned inference on 27 test images...')

for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0

    cx, cy = centroid_click(gt_mask)

    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        ft_predictor.set_image(img_rgb)
        preds, scores, _ = ft_predictor.predict(
            point_coords=np.array([[cx, cy]]),
            point_labels=np.array([1]),
            multimask_output=False
        )

    pred_mask = preds[0]
    stem = int(os.path.splitext(os.path.basename(img_path))[0])
    ft_results.append({
        'stem': stem,
        'dice': dice_score(pred_mask, gt_mask),
        'iou':  iou_score(pred_mask, gt_mask),
        'pred': pred_mask,
        'gt':   gt_mask,
        'img':  img_rgb,
    })

ft_dice = [r['dice'] for r in ft_results]
ft_iou  = [r['iou']  for r in ft_results]
print(f'\nFine-Tuned Results on {len(ft_results)} test images:')
print(f'  Dice: {np.mean(ft_dice):.3f} ± {np.std(ft_dice):.3f}')
print(f'  IoU:  {np.mean(ft_iou):.3f} ± {np.std(ft_iou):.3f}')

## Cell 11 — Results Comparison

Dice and IoU for every test image, plus a side-by-side visual panel.  
This is the first time in this project we have real numbers on held-out data.

In [ ]:
# Per-image table
print(f"{'Stem':>6} | {'ZS Dice':>8} | {'FT Dice':>8} | {'ZS IoU':>7} | {'FT IoU':>7} | Winner")
print('-'*60)
zs_wins = ft_wins = ties = 0
for zr, fr in zip(sorted(zs_results, key=lambda r: r['stem']),
                  sorted(ft_results, key=lambda r: r['stem'])):
    assert zr['stem'] == fr['stem']
    winner = 'ZS' if zr['dice'] > fr['dice'] else ('FT' if fr['dice'] > zr['dice'] else 'tie')
    if winner == 'ZS': zs_wins += 1
    elif winner == 'FT': ft_wins += 1
    else: ties += 1
    print(f"{zr['stem']:>6} | {zr['dice']:>8.3f} | {fr['dice']:>8.3f} | {zr['iou']:>7.3f} | {fr['iou']:>7.3f} | {winner}")

print('='*60)
print(f"MEAN   | {np.mean(zs_dice):>8.3f} | {np.mean(ft_dice):>8.3f} | {np.mean(zs_iou):>7.3f} | {np.mean(ft_iou):>7.3f}")
print(f"STD    | {np.std(zs_dice):>8.3f} | {np.std(ft_dice):>8.3f} | {np.std(zs_iou):>7.3f} | {np.std(ft_iou):>7.3f}")
print(f"\nZero-shot wins: {zs_wins} | Fine-tuned wins: {ft_wins} | Ties: {ties}")

In [ ]:
# Visual comparison: 6 test images side by side
zs_by_stem = {r['stem']: r for r in zs_results}
ft_by_stem = {r['stem']: r for r in ft_results}
stems_show = sorted(zs_by_stem.keys())[:6]

fig, axes = plt.subplots(4, 6, figsize=(24, 16))
row_labels = ['Original','Ground Truth','Zero-Shot (Red)','Fine-Tuned (Blue)']
for col, stem in enumerate(stems_show):
    zr = zs_by_stem[stem]; fr = ft_by_stem[stem]
    img = zr['img'][:,:,0]
    axes[0,col].imshow(img, cmap='gray'); axes[0,col].set_title(f'#{stem}'); axes[0,col].axis('off')
    axes[1,col].imshow(zr['gt'], cmap='gray'); axes[1,col].axis('off')
    axes[2,col].imshow(img, cmap='gray'); axes[2,col].imshow(zr['pred'], alpha=0.45, cmap='Reds')
    axes[2,col].set_title(f'D={zr["dice"]:.2f}'); axes[2,col].axis('off')
    axes[3,col].imshow(img, cmap='gray'); axes[3,col].imshow(fr['pred'], alpha=0.45, cmap='Blues')
    axes[3,col].set_title(f'D={fr["dice"]:.2f}'); axes[3,col].axis('off')
for row, label in enumerate(row_labels):
    axes[row,0].set_ylabel(label, fontsize=10)
plt.suptitle(f'MedSAM2 on DCA1 Test Split | ZS Dice={np.mean(zs_dice):.3f} | FT Dice={np.mean(ft_dice):.3f}', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/dca1_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
!gsutil cp /tmp/dca1_comparison.png {BUCKET}/results/dca1_comparison.png
print('Results saved to GCS')

## Next Steps

**If fine-tuned > zero-shot:**  
Fine-tuning on DCA1 is beneficial even with 107 images. Next: increase epochs, try LoRA on the encoder, try ARCADE dataset (1,500 images).

**If zero-shot > fine-tuned:**  
The 107 DCA1 images are not enough to overcome MedSAM2's general capability. Best path forward: pseudo-label fine-tuning using zero-shot predictions on CoronaryDominance video as training signal.

**The contrast detector idea:**  
Add a per-frame contrast quality score (vessel region brightness variance) to the video pipeline.  
When score drops (contrast washout), suppress propagation. When score rises (new injection), re-prompt.  
This is the most original contribution in this project and requires zero additional annotations.

---
## Experiment 2 — Box + Click Prompt (Augmented, 50 Epochs)

**Why boxes help:**  
A point tells the decoder *where* to click but not *how large* the object is.  
A bounding box gives it a hard spatial constraint — the mask must live inside the box.  
Combined with a centroid click, the decoder gets both a size prior and a location prior.

**Box derivation:**  
From the GT mask: `[min_x, min_y, max_x, max_y]` + 5% padding on each side.  
At inference we use the GT-derived box — this simulates a user who drew a rough box around the vessel tree.

**Data augmentation:**  
Random horizontal flip, brightness/contrast jitter, Gaussian noise.  
This is the single fastest way to push Dice up — effectively multiplies the training set.

**Target: 0.85 Dice.**

In [ ]:
import torchvision.transforms.functional as TF
import random as rnd

def get_box_from_mask(mask_np, pad_frac=0.05):
    """Bounding box of GT mask with padding. Returns [x1,y1,x2,y2] in original image space."""
    ys, xs = np.where(mask_np > 0)
    if len(ys) == 0:
        h, w = mask_np.shape
        return np.array([0, 0, w, h], dtype=np.float32)
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    h, w = mask_np.shape
    pad_x = max(1, int((x2 - x1) * pad_frac))
    pad_y = max(1, int((y2 - y1) * pad_frac))
    x1 = max(0, x1 - pad_x);  y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x);  y2 = min(h, y2 + pad_y)
    return np.array([x1, y1, x2, y2], dtype=np.float32)

class DCA1DatasetAug(Dataset):
    """DCA1 dataset with augmentation and box+click prompts."""
    def __init__(self, img_paths, mask_paths, augment=True):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img      = Image.open(self.img_paths[idx]).convert('RGB')
        mask_pil = Image.open(self.mask_paths[idx]).convert('L')
        mask_np  = np.array(mask_pil)  # 300×300 original

        # ── Augmentation ──────────────────────────────────────────────────
        if self.augment:
            if rnd.random() > 0.5:
                img      = TF.hflip(img)
                mask_pil = TF.hflip(mask_pil)
                mask_np  = np.array(mask_pil)
            if rnd.random() > 0.5:
                img = TF.vflip(img)
                mask_pil = TF.vflip(mask_pil)
                mask_np  = np.array(mask_pil)
            # Brightness + contrast jitter (X-ray specific: keep mean gray level)
            img = TF.adjust_brightness(img, 0.7 + rnd.random() * 0.6)
            img = TF.adjust_contrast(img,   0.7 + rnd.random() * 0.6)
            # Gaussian noise
            img_arr = np.array(img).astype(np.float32)
            img_arr += np.random.normal(0, 5, img_arr.shape)
            img_arr = np.clip(img_arr, 0, 255).astype(np.uint8)
            img = Image.fromarray(img_arr)

        # ── Normalise (match SAM2 pretrained encoder preprocessing) ───────
        img_t = transforms.Compose([
            transforms.Resize((MODEL_SIZE, MODEL_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])(img)

        # ── Target mask at 256×256 ─────────────────────────────────────────
        mask_256 = np.array(mask_pil.resize((256,256), Image.NEAREST))
        mask_t   = torch.from_numpy((mask_256 > 0).astype(np.float32)).unsqueeze(0)

        # ── Click: random foreground pixel ─────────────────────────────────
        ys, xs = np.where(mask_np > 0)
        if len(ys) > 0:
            pick = np.random.randint(len(ys))
            py, px = int(ys[pick]), int(xs[pick])
        else:
            py, px = mask_np.shape[0]//2, mask_np.shape[1]//2
        h, w = mask_np.shape
        point = torch.tensor([px/w*MODEL_SIZE, py/h*MODEL_SIZE], dtype=torch.float32)

        # ── Box: GT mask bounds + 5% padding, scaled to MODEL_SIZE ─────────
        box_orig = get_box_from_mask(mask_np, pad_frac=0.05)  # [x1,y1,x2,y2] in 300-space
        box_scaled = torch.tensor([
            box_orig[0]/w*MODEL_SIZE, box_orig[1]/h*MODEL_SIZE,
            box_orig[2]/w*MODEL_SIZE, box_orig[3]/h*MODEL_SIZE
        ], dtype=torch.float32)   # [x1,y1,x2,y2] in 512-space

        return img_t, mask_t, point, box_scaled

train_aug_ds  = DCA1DatasetAug(train_img_paths, train_mask_paths, augment=True)
train_aug_loader = DataLoader(train_aug_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Augmented train dataset: {len(train_aug_ds)} images, {len(train_aug_loader)} batches/epoch')

### Training Loop — Box + Click

SAM2's prompt encoder accepts **simultaneous** point + box prompts.  
We pass both at once — the encoder combines them into a single set of sparse embeddings.  
The box embedding gives the decoder a hard spatial prior; the click identifies the specific vessel.

In [ ]:
CKPT_PATH_BOX = '/tmp/medsam2_dca1_boxclick.pt'
EPOCHS_BOX    = 20   # more epochs with augmentation

# Fresh model from base checkpoint
model_box = build_sam2(CONFIG, CHECKPOINT, device=device)
for name, param in model_box.named_parameters():
    param.requires_grad = ('image_encoder' not in name)

optimizer_box = optim.AdamW([p for p in model_box.parameters() if p.requires_grad],
                             lr=5e-5, weight_decay=1e-4)
scheduler_box = optim.lr_scheduler.CosineAnnealingLR(optimizer_box, T_max=EPOCHS_BOX)
scaler_box    = GradScaler()

best_loss_box = float('inf')
losses_box    = []

print(f'Training box+click model — {EPOCHS_BOX} epochs')

for epoch in range(EPOCHS_BOX):
    model_box.train()
    epoch_loss = 0.0

    for imgs, masks, points, boxes in tqdm(train_aug_loader, desc=f'Ep {epoch+1}/{EPOCHS_BOX}', leave=False):
        optimizer_box.zero_grad()
        B = imgs.shape[0]
        imgs_dev   = imgs.to(device)
        points_dev = points.to(device)
        boxes_dev  = boxes.to(device)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            with torch.no_grad():
                backbone_out = model_box.forward_image(imgs_dev)
                _, vision_feats, _, _ = model_box._prepare_backbone_features(backbone_out)
                if model_box.directly_add_no_mem_embed:
                    vision_feats[-1] = vision_feats[-1] + model_box.no_mem_embed

            feats = [
                feat.permute(1,2,0).view(B,-1,*fs)
                for feat, fs in zip(vision_feats[::-1], BB_FEAT_SIZES[::-1])
            ][::-1]
            image_embed    = feats[-1]
            high_res_feats = feats[:-1]

            all_logits = []
            for i in range(B):
                pt       = points_dev[i].reshape(1,1,2)
                pt_label = torch.ones(1,1, dtype=torch.int, device=device)
                # Box: SAM2 expects (1, 4) — [x1, y1, x2, y2]
                box = boxes_dev[i].reshape(1,4)

                sparse_emb, dense_emb = model_box.sam_prompt_encoder(
                    points=(pt, pt_label),
                    boxes=box,       # <── box prompt added
                    masks=None
                )
                low_res_masks, _, _, _ = model_box.sam_mask_decoder(
                    image_embeddings=image_embed[i].unsqueeze(0),
                    image_pe=model_box.sam_prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_emb,
                    dense_prompt_embeddings=dense_emb,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=[f[i].unsqueeze(0) for f in high_res_feats],
                )
                all_logits.append(low_res_masks)

            logits  = torch.cat(all_logits, dim=0)
            targets = masks.to(device)
            if targets.shape[-2:] != logits.shape[-2:]:
                targets = F.interpolate(targets.float(), size=logits.shape[-2:], mode='nearest')
            loss = combined_loss(logits, targets, pos_weight)

        scaler_box.scale(loss).backward()
        scaler_box.unscale_(optimizer_box)
        torch.nn.utils.clip_grad_norm_([p for p in model_box.parameters() if p.requires_grad], 1.0)
        scaler_box.step(optimizer_box)
        scaler_box.update()
        epoch_loss += loss.item()

    epoch_loss /= len(train_aug_loader)
    losses_box.append(epoch_loss)
    scheduler_box.step()
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | Loss: {epoch_loss:.4f}')
    if epoch_loss < best_loss_box:
        best_loss_box = epoch_loss
        torch.save(model_box.state_dict(), CKPT_PATH_BOX)

print(f'\nDone. Best loss: {best_loss_box:.4f}')
!gsutil cp {CKPT_PATH_BOX} {BUCKET}/checkpoints/medsam2_dca1_boxclick.pt

plt.figure(figsize=(8,4))
plt.plot(losses_box, label='Train Loss (box+click, augmented)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig('/tmp/ft_boxclick_loss.png', dpi=100)
plt.show()

### Box + Click Inference on Test Split

At inference, we derive the bounding box from the GT mask (same as training).  
This simulates a user who drew a rough box around the vessel region and then clicked on a vessel.

In [ ]:
# Load box+click fine-tuned model
model_box_inf = build_sam2(CONFIG, CHECKPOINT, device=device)
incomp = model_box_inf.load_state_dict(torch.load(CKPT_PATH_BOX, map_location=device), strict=False)
print(f'Missing keys: {len(incomp.missing_keys)} | Unexpected: {len(incomp.unexpected_keys)}')
assert len(incomp.missing_keys) == 0, 'Checkpoint load failed — keys missing!'
print('✓ Clean checkpoint load')
model_box_inf.eval()
box_predictor = SAM2ImagePredictor(model_box_inf)

box_results = []
print('Running box+click inference on 27 test images...')

for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0

    cx, cy  = centroid_click(gt_mask)
    box_np  = get_box_from_mask(gt_mask, pad_frac=0.05)  # [x1,y1,x2,y2]

    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        box_predictor.set_image(img_rgb)
        preds, scores, _ = box_predictor.predict(
            point_coords=np.array([[cx, cy]]),
            point_labels=np.array([1]),
            box=box_np[None],           # SAM2 predictor expects (1, 4)
            multimask_output=False
        )

    pred_mask = preds[0]
    stem = int(os.path.splitext(os.path.basename(img_path))[0])
    box_results.append({
        'stem': stem,
        'dice': dice_score(pred_mask, gt_mask),
        'iou':  iou_score(pred_mask, gt_mask),
        'pred': pred_mask,
        'gt':   gt_mask,
        'img':  img_rgb,
    })

box_dice = [r['dice'] for r in box_results]
box_iou  = [r['iou']  for r in box_results]
print(f'\n=== Final Comparison ===')
print(f'Zero-Shot (click only):       Dice={np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU={np.mean(zs_iou):.3f}')
print(f'Fine-Tuned (click, 20ep):     Dice={np.mean(ft_dice):.3f} ± {np.std(ft_dice):.3f} | IoU={np.mean(ft_iou):.3f}')
print(f'Fine-Tuned (box+click, 20ep): Dice={np.mean(box_dice):.3f} ± {np.std(box_dice):.3f} | IoU={np.mean(box_iou):.3f}')

In [ ]:
# Three-way visual comparison on 5 test images
stems_show5 = sorted([r['stem'] for r in box_results])[:5]
zs_s  = {r['stem']: r for r in zs_results}
ft_s  = {r['stem']: r for r in ft_results}
bx_s  = {r['stem']: r for r in box_results}

fig, axes = plt.subplots(4, 5, figsize=(22, 16))
row_labels = ['Original + GT overlay', 'Zero-Shot (click)', 'Fine-Tuned (click, 20ep)', 'Fine-Tuned (box+click, 50ep)']

for col, stem in enumerate(stems_show5):
    zr = zs_s[stem]; fr = ft_s[stem]; br = bx_s[stem]
    img = zr['img'][:,:,0]

    axes[0,col].imshow(img, cmap='gray')
    axes[0,col].imshow(zr['gt'], alpha=0.3, cmap='Greens')
    axes[0,col].set_title(f'#{stem}'); axes[0,col].axis('off')

    axes[1,col].imshow(img, cmap='gray')
    axes[1,col].imshow(zr['pred'], alpha=0.45, cmap='Reds')
    axes[1,col].set_title(f'D={zr["dice"]:.2f}'); axes[1,col].axis('off')

    axes[2,col].imshow(img, cmap='gray')
    axes[2,col].imshow(fr['pred'], alpha=0.45, cmap='Blues')
    axes[2,col].set_title(f'D={fr["dice"]:.2f}'); axes[2,col].axis('off')

    axes[3,col].imshow(img, cmap='gray')
    axes[3,col].imshow(br['pred'], alpha=0.45, cmap='Purples')
    axes[3,col].set_title(f'D={br["dice"]:.2f}'); axes[3,col].axis('off')

for row, label in enumerate(row_labels):
    axes[row,0].set_ylabel(label, fontsize=9)

plt.suptitle('Three-Way Comparison on DCA1 Test Split', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/dca1_threeway.png', dpi=120, bbox_inches='tight')
plt.show()
!gsutil cp /tmp/dca1_threeway.png {BUCKET}/results/dca1_threeway.png
print('Saved')